# Calling LLMs: A Step-by-Step Approach

In this Jupyter notebook, we will dive deep into the world of LLMs and demonstrate how to use them effectively. First, we will explore how to deploy an LLM, and then we will see how to connect to it.


## Step 1: Install the Necessary Libraries

In [1]:
!pip install requests

In [2]:
!pip install openai

  Using cached idna-3.20-py3-none-any.whl.metadata (7.2 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 19.3 MB/s  0:00:00
Using cached idna-3.20-py3-none-any.whl (69 kB)
  Attempting uninstall: idna
    Found existing installation: idna 3.11
    Uninstalling idna-3.11:
      Successfully uninstalled idna-3.11
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5/5 [openai]2m4/5 [openai]
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
tensorflow 2.17.1 requires keras>=3.2.0, but you have keras 2.15.0 which is incompatible.
aioli-sdk 1.12.0rc0.dev42+gd9a6d6e46.dirty.ldev requires urllib3<2.3.0,>=2.0.0, but you have urllib3 1.26.20 which is incompatible.


In [3]:
import requests
import subprocess
import json
from openai import OpenAI

## Step 2: Configure the Model Pod (only appliable for Model Manager deployments)

NOTE: unless you are using [Method C: using Model Manager](deployment_with_model_manager.md) explained in the README for Model Deployment, you can skip this step.

In [4]:
!kubectl get pods

NAME                                                              READY   STATUS    RESTARTS      AGE
default-notebook-0                                                2/2     Running   0             19d
digitrec-serve-predictor-00001-deployment-7868f744c6-r6p28        2/2     Running   0             19d
fs-66cb4b7695-s6mdr                                               2/2     Running   0             19d
llama-3-3-nemotron-super-49b-v1-5-predictor-00001-deployme2kp8w   4/4     Running   0             19d
ml-pipeline-ui-artifact-6f9ccf5878-7sdtg                          2/2     Running   0             19d
nv-embedqa-e5-v5-predictor-00001-deployment-5b5cf9d9d4-sgmrk      4/4     Running   1 (19d ago)   19d
pipeline-tester-pipeline-tester-6fddbb8fc-x84rq                   2/2     Running   0             19d
pipeline-tester-pipeline-tester-runner-pipeline-1-6c9c777c5f5zc   2/2     Running   0             19d
pipeline-tester-pipeline-tester-runner-pipeline-2-59fc785899jsc   2/2     Running 

Now, ensure you have the 'access token'. You should see in the lables the line **hpe-ezua/add-auth-token=true**. See the documentation for more information. 

## Step 3: Obtaining the Auth Token

The secret mounts the notebook's filesystem in the local pod at **/etc/secrets/ezua/.auth_token**. The notebook reads the token from the file and puts the token in an outgoing request. The notebook can do this repeatedly because the token service in Unified Analytics regularly refreshes the access token.

In [5]:
secret_file_path = "/etc/secrets/ezua/.auth_token"

with open(secret_file_path, "r") as file:
    token = file.read().strip()

print(f"Authorization Token: {token}")

Authorization Token: eyJhbGciOiJSUzI1NiIsInR5cCIgOiAiSldUIiwia2lkIiA6ICJORlA1VDdJWWhUNnd1cS1aMS1zT3piaWNwNVRmSFdwc0pGejJmN3hFRGU0In0.eyJleHAiOjE3OTEzODczNjgsImlhdCI6MTc5MTM4NTU2OCwiYXV0aF90aW1lIjoxNzkxMzg1NDE5LCJqdGkiOiJkYzVmZmY5Ny03MmU1LTRiMTgtOTJlMi1mM2M4NmYxYmQyNzgiLCJpc3MiOiJodHRwczovL2tleWNsb2FrLnAyYWIxMDY2NTUtYWktYXBwbGljYXRpb24uZGMwMi5pdHMuaHBlY29ycC5uZXQvcmVhbG1zL1VBIiwic3ViIjoiN2RjZTA3YzMtYmIxNy00NTU0LWFhNjgtMDk2OTRjYjc1MGE1IiwidHlwIjoiQmVhcmVyIiwiYXpwIjoidWEiLCJub25jZSI6IjNtaUVPeEJLVktwRlp4YzFCbXFVMk9KOElsS3NNc2ZJMWIySk41QnNyXzgiLCJzZXNzaW9uX3N0YXRlIjoiMmJhNWIxODUtYzc0Mi00MTVhLThlNmQtMzBmNjgzNjMzMDkzIiwiYWNyIjoiMCIsInNjb3BlIjoib3BlbmlkIHByb2ZpbGUgb2ZmbGluZV9hY2Nlc3MgZW1haWwgcDJhYjEwNjY1NTp1YSIsInNpZCI6IjJiYTViMTg1LWM3NDItNDE1YS04ZTZkLTMwZjY4MzYzMzA5MyIsInVpZCI6IjEwMDAwMDA0IiwiZW1haWxfdmVyaWZpZWQiOmZhbHNlLCJnaWQiOiIxMDAxIiwibmFtZXNwYWNlIjoicHJvamVjdC11c2VyLXN1cmFqLXItcy1leHQiLCJuYW1lIjoiU3VyYWogUiBTIiwiZ3JvdXBzIjpbInVhLWVuYWJsZWQiLCJvZmZsaW5lX2FjY2VzcyIsImFkbWluIiwidW1hX2F1dGh


### NOTE: 
*This token has a short lifespan (10 minutes). If you encounter the error 'JWT expired,' please run this step and the curl command again.*


## Step 4: Model Interaction

Retrieve the URL of the inference service you are going to use, e.g.:

In [6]:
!kubectl get isvc

NAME                                URL                                                                                                                           READY   PREV   LATEST   PREVROLLEDOUTREVISION   LATESTREADYREVISION                                 AGE
digitrec-serve                      https://digitrec-serve.project-user-suraj-r-s-ext.serving.p2ab106655-ai-application.dc02.its.hpecorp.net                      True           100                              digitrec-serve-predictor-00001                      19d
llama-3-3-nemotron-super-49b-v1-5   https://llama-3-3-nemotron-super-49b-v1-5.project-user-suraj-r-s-ext.serving.p2ab106655-ai-application.dc02.its.hpecorp.net   True           100                              llama-3-3-nemotron-super-49b-v1-5-predictor-00001   19d
nv-embedqa-e5-v5                    https://nv-embedqa-e5-v5.project-user-suraj-r-s-ext.serving.p2ab106655-ai-application.dc02.its.hpecorp.net                    True           100                      

NOTE: 
If you are using deployment_with_model_manager file, you can find it in the section **Tools & Frameworks > Data Engineering > Model Manager > Manage > InferenceService > URL**.    
For more information, please refer to the documentation.

In fhe following cell, replace the value of the `url` variable with the desired inference URL found above:

In [8]:
url = "https://llama-3-3-nemotron-super-49b-v1-5.project-user-suraj-r-s-ext.serving.p2ab106655-ai-application.dc02.its.hpecorp.net"
endpoint = f'{url}/v1/completions'

### Example 1: Using the curl command. 

Curl is a command-line tool used to trasnfer data to or from server using various protocols, including HTTP. We can interacting with web service. To use curl, you need to include specific headers:

- accept: application/json: You expect a response in JSON format.
- Authorization: Bearer {your_token}: Your actual token.
- Content-Type: application/json: The type of data you are sendin to the server,which in this case is JSON. 
- Configuration of the instruction: This involves specifying the command option: for example we are using the model llama3-8b-instruction.


In [9]:
curl_command = [
    'curl',
    '-H','accept: application/json',
    '-H', f'Authorization: Bearer {token}', 
    '-H', 'Content-Type: application/json',
    '-d', '{"model": "nvidia/llama-3.3-nemotron-super-49b-v1.5", "prompt": "Once upon a time", "max_tokens": 64}',
    '-k',  # Skip SSL verification
    endpoint
]

result = subprocess.run(curl_command, capture_output=True, text=True)

In [10]:
# Assuming result.stdout contains the server's JSON response
response = json.loads(result.stdout)

# Check if the response was successful
if 'choices' in response and len(response['choices']) > 0:
    print("Response successful.")
    # Extract and display the generated text
    print("-----Generated text:---------------")
    generated_text = response['choices'][0]['text']
    print(generated_text.strip())
else:
    print("Response not successful.")
    # You can display more information if necessary
    if 'detail' in response:
        print("Error details:", response['detail'])
    else:
        print("Server response:", response)

Response successful.
-----Generated text:---------------
, in a village called Koral, there lived a beautiful young woman named Lira. Lira was known for her kindness and compassion, often helping those in need without expecting anything in return. However, despite her good heart, Lira was also incredibly stubborn and had a tendency to follow her own path, even when


## Example 2: Basic Text Generation. 

We have our API set up so let's explore basic text generation with our LLM. Basic Text Generation in a Large Language Model (LLM) refers to the capability of these models to produce coherent and contextually relevant text based on the input they receive.

In [11]:
headers = {
    "accept": "application/json",
    "Content-Type": "application/json",
    "Authorization": f"Bearer {token}"
}

In [12]:
def call_api(prompt, api_key=token):
    try:
        # Initialize the OpenAI client
        client = OpenAI(
            base_url=endpoint,
            api_key=api_key
        )

        # Create the completion request
        completion = client.chat.completions.create(
            model= "nvidia/llama-3.3-nemotron-super-49b-v1.5",
            messages=[{"role": "user", "content": prompt}],
            temperature=0.5,
            top_p=0.7,
            max_tokens=500,
            stream=True
        )

        # Collect and return the full response
        response_text = ""
        for chunk in completion:
            if chunk.choices[0].delta.content is not None:
                response_text += chunk.choices[0].delta.content

        return response_text
    except Exception as e:
        print(f"An exception occurred: {e}")
        print(f"Looks like the User has the wrong access Token")
        if str(e) == "RBAC: access denied":
            return f"Looks like your user has the wrong access token and isn't privileged to access the model"
        else:
            return f"The fronted couldn't establish a connection to the model at the following URL: {endpoint}."

In [13]:
def generate_text(prompt, max_tokens=100):
    """Generate text using the LLM API.
    
    Args:
        prompt (str): The input prompt for the LLM.
        max_tokens (int): Maximum number of tokens to generate.
    
    Returns:
        str: The generated text or an error message.
    """
    data = {
        "model": "nvidia/llama-3.3-nemotron-super-49b-v1.5",
        "prompt": prompt,
        "max_tokens": max_tokens
    }
    
    response = requests.post(endpoint, headers=headers, json=data, verify=False)  # Skip SSL verification
    
    if response.status_code == 200:
        return response.json()["choices"][0]["text"].strip()
    else:
        return f"Error: {response.status_code} - {response.text}"

In [14]:
# Test the API connection
test_prompt = "Hello world!"
print("---------------------------")
print(generate_text(test_prompt))

---------------------------


/opt/conda/lib/python3.11/site-packages/urllib3/connectionpool.py:1064: InsecureRequestWarning: Unverified HTTPS request is being made to host 'llama-3-3-nemotron-super-49b-v1-5.project-user-suraj-r-s-ext.serving.p2ab106655-ai-application.dc02.its.hpecorp.net'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/1.26.x/advanced-usage.html#ssl-warnings
  warnings.warn(


Meet Skye, a happy-go-lucky golden retriever who’s ready to embark on adventures and make new friends. Skye loves exploring the outdoors, chasing butterflies, and fetching sticks in the park. She’s curious and playful, always eager to learn and please. With her shiny coat and wagging tail, Skye is the ultimate companion for anyone looking for a loyal friend. Skye is currently located in Sunny California and is ready to bring joy and excitement into your life. Don’t miss


## Example 3: Text Summarization

Text summarization is the task of creating a concise and coherent summary of a longer text while preserving its key information. LLMs can be quite effective at this task when given appropriate prompts. Let's try summarizing a paragraph about the Industrial Revolution:

In [15]:
summarization_prompt = """
Summarize the following paragraph in one or two sentences:

The Renaissance, which flourished from the 14th to the 17th centuries, was a period marked by a revival of interest in art, science, and classical antiquity in Europe. Originating in Italy, this cultural movement led to significant developments in painting, literature, and humanism, fostering creativity and innovation that laid the groundwork for the modern world.

Summary:
"""

print("Text Summarization result:")
print("---------------------------")
print(generate_text(summarization_prompt, max_tokens=50))

Text Summarization result:
---------------------------


/opt/conda/lib/python3.11/site-packages/urllib3/connectionpool.py:1064: InsecureRequestWarning: Unverified HTTPS request is being made to host 'llama-3-3-nemotron-super-49b-v1-5.project-user-suraj-r-s-ext.serving.p2ab106655-ai-application.dc02.its.hpecorp.net'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/1.26.x/advanced-usage.html#ssl-warnings
  warnings.warn(


The Renaissance was a period in Europe from the 14th to 17th centuries that revived interest in art, science, and classical antiquity, starting in Italy and driving major developments in various fields that shaped the modern world.
</summary>


## Example 4: Question Answering

Question Answeting is the task of to understand and respond to questions posed by users. This involves analyzing the input question, retrieving relevant information from their training data, and generating an accurate and coherent response.

In [16]:
qa_prompt = """
Answer the following question based on the given context:

Context: A small town in the countryside is preparing for its annual harvest festival. The locals are excited about the event, which features a parade, food stalls with local produce, and various competitions like pumpkin carving and pie-eating contests. Everyone is involved in the preparations, from decorating the town square to baking pies and organizing games for children.

Question: What are some traditional activities that townspeople might include in a harvest festival, and how do these activities reflect the community's culture?

Answer:
"""

print("Question Answering result:")
print("---------------------------")
print(generate_text(qa_prompt, max_tokens=50))

Question Answering result:
---------------------------


/opt/conda/lib/python3.11/site-packages/urllib3/connectionpool.py:1064: InsecureRequestWarning: Unverified HTTPS request is being made to host 'llama-3-3-nemotron-super-49b-v1-5.project-user-suraj-r-s-ext.serving.p2ab106655-ai-application.dc02.its.hpecorp.net'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/1.26.x/advanced-usage.html#ssl-warnings
  warnings.warn(


The paragraph describes a small town preparing for its annual harvest festival, which includes activities such as a parade, food stalls, competitions like pumpkin carving and pie-eating contests, and games for children. These activities reflect the community's culture in several ways.


# Example 5: Text simplification

Text simplification is the process of transforming complex or dense text into clearer, more accessible language.

In [ ]:
simplification_prompt = """
Simplify the following paragraph by using simpler language and making it easier to understand: .. 
Global warming is a long-term increase in Earth's average surface temperature due to human activities, primarily the burning of fossil fuels, which increases the levels of greenhouse gases in the atmosphere. This phenomenon has far-reaching effects on climate patterns, leading to more severe weather events, rising sea levels, and disruptions to ecosystems.
Simplified Text:

"""
print("Text simplification result:")
print("---------------------------")
print(generate_text(simplification_prompt, max_tokens=50))

Text simplification result:
---------------------------


/opt/conda/lib/python3.11/site-packages/urllib3/connectionpool.py:1064: InsecureRequestWarning: Unverified HTTPS request is being made to host 'llama-3-3-nemotron-super-49b-v1-5.project-user-suraj-r-s-ext.serving.p2ab106655-ai-application.dc02.its.hpecorp.net'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/1.26.x/advanced-usage.html#ssl-warnings
  warnings.warn(
